# 18. Desired state, idempotency, and rollback

**Level:** Advanced

**Learning outcome:** Separate intent, observation, plan, approval, application, verification, and recovery.

**Prerequisites:** 03-04 and 13-17: validation, resource lifetime, failures, and workflows.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](17_Observer_Command_Template_Registry_and_Composite.ipynb) | [Next notebook](19_Testing_Fakes_Mocks_and_Invariants.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Advanced

**Learning goal:** Plan differences, preserve idempotency, simulate rollback, and keep approvals visible.

**Intent, observation, and plan are separate things.** Intent describes what should be true.
Observation describes what was read at a particular time. A plan describes the proposed
transition. A successful command response is not proof that intent is now satisfied; collect
fresh observations and verify postconditions.

**Idempotency** means repeating an operation has the same final intended effect as performing
it once. A set-style assignment can be idempotent; incrementing a value usually is not.
Reconciliation should produce no changes when observed and desired state already match.
Normalize semantically equivalent data before comparing it.

**Dry run** must not execute writes. A plan should be reviewable before execution and bound to
the exact target, intended change, and baseline. Approval is distinct from validation. These
lessons use boolean approvals only to teach control flow, not as real authorization.

**Transactions and rollback.** A local dictionary can be copied and restored atomically from
one process's point of view, but a multi-device rollout is not a single Python transaction.
Devices may have candidate configurations or commit-confirmed features with their own rules.
A rollback can also fail. Production needs prechecks, backups, staged rollout, postchecks,
drift/concurrency controls, durable audit records, and explicit partial-failure reporting.

**Audit records** should say what was planned, what was attempted, what succeeded, and what
failed, while excluding credentials and unnecessary sensitive configuration. A list of
in-memory events here illustrates semantics but is neither durable nor tamper-evident.

**Pause and explain:** State the difference between a dry-run plan, an approved change, and verified desired state.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 061: Desired versus observed diff

**Core concept:** Produce a deterministic change map only for differing fields.

**Network-engineering purpose:** A configuration object can report exactly which intended fields need reconciliation.

### Read the implementation in this order
1. Copy desired fields into the intent.
2. Compare only fields managed by this intent.
3. Record old and new values for changes.
4. Check equality produces an empty plan.

**Common mistake / boundary:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

**Practice after the ten worked scenarios:** Introduce an explicit missing sentinel for a model where None is a legitimate desired value.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0601 - Desired versus observed diff: branch scenario

**Scenario 01/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [1]:
# Example 0601: Desired versus observed diff - branch
case = {'mtu': 1500}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{}


### Example 0602 - Desired versus observed diff: campus scenario

**Scenario 02/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 9000)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [2]:
# Example 0602: Desired versus observed diff - campus
case = {'mtu': 9000}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 9000)}


### Example 0603 - Desired versus observed diff: dc scenario

**Scenario 03/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 9216)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [3]:
# Example 0603: Desired versus observed diff - dc
case = {'mtu': 9216}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 9216)}


### Example 0604 - Desired versus observed diff: wan scenario

**Scenario 04/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 576)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [4]:
# Example 0604: Desired versus observed diff - wan
case = {'mtu': 576}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 576)}


### Example 0605 - Desired versus observed diff: lab scenario

**Scenario 05/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 1600)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [5]:
# Example 0605: Desired versus observed diff - lab
case = {'mtu': 1600}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 1600)}


### Example 0606 - Desired versus observed diff: wireless scenario

**Scenario 06/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [6]:
# Example 0606: Desired versus observed diff - wireless
case = {'mtu': 1500}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{}


### Example 0607 - Desired versus observed diff: remote scenario

**Scenario 07/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 2000)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [7]:
# Example 0607: Desired versus observed diff - remote
case = {'mtu': 2000}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 2000)}


### Example 0608 - Desired versus observed diff: core scenario

**Scenario 08/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 4470)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [8]:
# Example 0608: Desired versus observed diff - core
case = {'mtu': 4470}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 4470)}


### Example 0609 - Desired versus observed diff: mgmt scenario

**Scenario 09/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 9100)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [9]:
# Example 0609: Desired versus observed diff - mgmt
case = {'mtu': 9100}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 9100)}


### Example 0610 - Desired versus observed diff: backup scenario

**Scenario 10/10; pattern 061; Advanced.** This is a self-contained offline exercise. A configuration object can report exactly which intended fields need reconciliation.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Copy desired fields into the intent. Check equality produces an empty plan. Follow the class-specific steps under Pattern 061 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'mtu': (1500, 8192)}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** observed.get treats absent and None alike; this toy model only manages non-None MTU values.

In [10]:
# Example 0610: Desired versus observed diff - backup
case = {'mtu': 8192}

class Intent:
    def __init__(self, desired):
        self.desired = dict(desired)

    def diff(self, observed):
        return {key: (observed.get(key), value)
                for key, value in self.desired.items() if observed.get(key) != value}

intent = Intent({"mtu": case["mtu"]})
plan = intent.diff({"mtu": 1500})
expected = {} if case["mtu"] == 1500 else {"mtu": (1500, case["mtu"])}
assert plan == expected
assert intent.diff({"mtu": case["mtu"]}) == {}
print(plan)


{'mtu': (1500, 8192)}


## Pattern 062: Idempotent local reconciliation

**Core concept:** Only write when the desired value differs from the current value.

**Network-engineering purpose:** Rerunning the same notebook cell should not count an unnecessary second change.

### Read the implementation in this order
1. Keep simulated state and a write counter.
2. Compare before assignment.
3. Apply the desired value twice.
4. Assert the second call reports no change and adds no write.

**Common mistake / boundary:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

**Practice after the ten worked scenarios:** Add a fresh observation after a simulated write and verify the resulting state.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0611 - Idempotent local reconciliation: branch scenario

**Scenario 01/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [11]:
# Example 0611: Idempotent local reconciliation - branch
case = {'mtu': 1500}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 0


### Example 0612 - Idempotent local reconciliation: campus scenario

**Scenario 02/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [12]:
# Example 0612: Idempotent local reconciliation - campus
case = {'mtu': 9000}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


### Example 0613 - Idempotent local reconciliation: dc scenario

**Scenario 03/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [13]:
# Example 0613: Idempotent local reconciliation - dc
case = {'mtu': 9216}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


### Example 0614 - Idempotent local reconciliation: wan scenario

**Scenario 04/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [14]:
# Example 0614: Idempotent local reconciliation - wan
case = {'mtu': 576}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


### Example 0615 - Idempotent local reconciliation: lab scenario

**Scenario 05/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [15]:
# Example 0615: Idempotent local reconciliation - lab
case = {'mtu': 1600}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


### Example 0616 - Idempotent local reconciliation: wireless scenario

**Scenario 06/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [16]:
# Example 0616: Idempotent local reconciliation - wireless
case = {'mtu': 1500}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 0


### Example 0617 - Idempotent local reconciliation: remote scenario

**Scenario 07/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [17]:
# Example 0617: Idempotent local reconciliation - remote
case = {'mtu': 2000}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


### Example 0618 - Idempotent local reconciliation: core scenario

**Scenario 08/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [18]:
# Example 0618: Idempotent local reconciliation - core
case = {'mtu': 4470}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


### Example 0619 - Idempotent local reconciliation: mgmt scenario

**Scenario 09/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [19]:
# Example 0619: Idempotent local reconciliation - mgmt
case = {'mtu': 9100}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


### Example 0620 - Idempotent local reconciliation: backup scenario

**Scenario 10/10; pattern 062; Advanced.** This is a self-contained offline exercise. Rerunning the same notebook cell should not count an unnecessary second change.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Keep simulated state and a write counter. Assert the second call reports no change and adds no write. Follow the class-specific steps under Pattern 062 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
writes: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Local idempotency does not resolve a real transport timeout with an unknown write outcome.

In [20]:
# Example 0620: Idempotent local reconciliation - backup
case = {'mtu': 8192}

class Reconciler:
    def __init__(self, current):
        self.current, self.writes = current, 0

    def ensure(self, desired):
        if self.current == desired:
            return False
        self.current = desired
        self.writes += 1
        return True

reconciler = Reconciler(1500)
changed = reconciler.ensure(case["mtu"])
assert changed == (case["mtu"] != 1500)
assert reconciler.ensure(case["mtu"]) is False
assert reconciler.writes == int(case["mtu"] != 1500)
print("writes:", reconciler.writes)


writes: 1


## Pattern 063: Dry-run plan cannot mutate state

**Core concept:** A planner can return proposed actions without executing them.

**Network-engineering purpose:** An engineer can inspect a requested MTU change before any simulated apply.

### Read the implementation in this order
1. Copy initial state.
2. Build a plan from the proposal.
3. Conditionally mutate only outside dry-run mode.
4. Check the state matches the selected mode.

**Common mistake / boundary:** Do not bury a write inside a method called plan or validate.

**Practice after the ten worked scenarios:** Separate plan and apply into different methods and require the exact plan object for apply.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0621 - Dry-run plan cannot mutate state: branch scenario

**Scenario 01/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `dry_run` | `True` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 1500} state: {'mtu': 1500}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [21]:
# Example 0621: Dry-run plan cannot mutate state - branch
case = {'mtu': 1500, 'dry_run': True}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 1500} state: {'mtu': 1500}


### Example 0622 - Dry-run plan cannot mutate state: campus scenario

**Scenario 02/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |
| `dry_run` | `False` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 9000} state: {'mtu': 9000}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [22]:
# Example 0622: Dry-run plan cannot mutate state - campus
case = {'mtu': 9000, 'dry_run': False}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 9000} state: {'mtu': 9000}


### Example 0623 - Dry-run plan cannot mutate state: dc scenario

**Scenario 03/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |
| `dry_run` | `True` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 9216} state: {'mtu': 1500}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [23]:
# Example 0623: Dry-run plan cannot mutate state - dc
case = {'mtu': 9216, 'dry_run': True}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 9216} state: {'mtu': 1500}


### Example 0624 - Dry-run plan cannot mutate state: wan scenario

**Scenario 04/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |
| `dry_run` | `False` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 576} state: {'mtu': 576}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [24]:
# Example 0624: Dry-run plan cannot mutate state - wan
case = {'mtu': 576, 'dry_run': False}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 576} state: {'mtu': 576}


### Example 0625 - Dry-run plan cannot mutate state: lab scenario

**Scenario 05/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |
| `dry_run` | `True` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 1600} state: {'mtu': 1500}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [25]:
# Example 0625: Dry-run plan cannot mutate state - lab
case = {'mtu': 1600, 'dry_run': True}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 1600} state: {'mtu': 1500}


### Example 0626 - Dry-run plan cannot mutate state: wireless scenario

**Scenario 06/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `dry_run` | `False` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 1500} state: {'mtu': 1500}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [26]:
# Example 0626: Dry-run plan cannot mutate state - wireless
case = {'mtu': 1500, 'dry_run': False}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan:

 {'mtu': 1500} state: {'mtu': 1500}


### Example 0627 - Dry-run plan cannot mutate state: remote scenario

**Scenario 07/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |
| `dry_run` | `True` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 2000} state: {'mtu': 1500}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [27]:
# Example 0627: Dry-run plan cannot mutate state - remote
case = {'mtu': 2000, 'dry_run': True}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 2000} state: {'mtu': 1500}


### Example 0628 - Dry-run plan cannot mutate state: core scenario

**Scenario 08/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |
| `dry_run` | `False` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 4470} state: {'mtu': 4470}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [28]:
# Example 0628: Dry-run plan cannot mutate state - core
case = {'mtu': 4470, 'dry_run': False}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 4470} state: {'mtu': 4470}


### Example 0629 - Dry-run plan cannot mutate state: mgmt scenario

**Scenario 09/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |
| `dry_run` | `True` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 9100} state: {'mtu': 1500}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [29]:
# Example 0629: Dry-run plan cannot mutate state - mgmt
case = {'mtu': 9100, 'dry_run': True}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 9100} state: {'mtu': 1500}


### Example 0630 - Dry-run plan cannot mutate state: backup scenario

**Scenario 10/10; pattern 063; Advanced.** This is a self-contained offline exercise. An engineer can inspect a requested MTU change before any simulated apply.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |
| `dry_run` | `False` |

**Logic to trace:** Copy initial state. Check the state matches the selected mode. Follow the class-specific steps under Pattern 063 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
plan: {'mtu': 8192} state: {'mtu': 8192}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not bury a write inside a method called plan or validate.

In [30]:
# Example 0630: Dry-run plan cannot mutate state - backup
case = {'mtu': 8192, 'dry_run': False}

class ChangeService:
    def __init__(self, state):
        self.state = dict(state)

    def run(self, desired, *, dry_run=True):
        plan = {"mtu": desired}
        if not dry_run:
            self.state.update(plan)
        return plan

service = ChangeService({"mtu": 1500})
plan = service.run(case["mtu"], dry_run=case["dry_run"])
assert plan == {"mtu": case["mtu"]}
assert service.state["mtu"] == (1500 if case["dry_run"] else case["mtu"])
print("plan:", plan, "state:", service.state)


plan: {'mtu': 8192} state: {'mtu': 8192}


## Pattern 064: Rollback a failed local transaction

**Core concept:** Snapshot state and restore it if a known postcondition fails.

**Network-engineering purpose:** A simulated rollout can illustrate why applying is not the final success criterion.

### Read the implementation in this order
1. Copy the baseline before mutation.
2. Apply proposed state.
3. Run an explicit postcheck.
4. Restore on the known verification error and propagate the error.

**Common mistake / boundary:** This restores one dictionary only; it does not provide network-wide atomicity.

**Practice after the ten worked scenarios:** Model rollback failure separately and report both the original and rollback errors.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0631 - Rollback a failed local transaction: branch scenario

**Scenario 01/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [31]:
# Example 0631: Rollback a failed local transaction - branch
case = {'mtu': 1500}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0632 - Rollback a failed local transaction: campus scenario

**Scenario 02/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [32]:
# Example 0632: Rollback a failed local transaction - campus
case = {'mtu': 9000}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0633 - Rollback a failed local transaction: dc scenario

**Scenario 03/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [33]:
# Example 0633: Rollback a failed local transaction - dc
case = {'mtu': 9216}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0634 - Rollback a failed local transaction: wan scenario

**Scenario 04/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [34]:
# Example 0634: Rollback a failed local transaction - wan
case = {'mtu': 576}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0635 - Rollback a failed local transaction: lab scenario

**Scenario 05/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [35]:
# Example 0635: Rollback a failed local transaction - lab
case = {'mtu': 1600}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0636 - Rollback a failed local transaction: wireless scenario

**Scenario 06/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [36]:
# Example 0636: Rollback a failed local transaction - wireless
case = {'mtu': 1500}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0637 - Rollback a failed local transaction: remote scenario

**Scenario 07/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [37]:
# Example 0637: Rollback a failed local transaction - remote
case = {'mtu': 2000}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0638 - Rollback a failed local transaction: core scenario

**Scenario 08/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [38]:
# Example 0638: Rollback a failed local transaction - core
case = {'mtu': 4470}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0639 - Rollback a failed local transaction: mgmt scenario

**Scenario 09/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [39]:
# Example 0639: Rollback a failed local transaction - mgmt
case = {'mtu': 9100}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


### Example 0640 - Rollback a failed local transaction: backup scenario

**Scenario 10/10; pattern 064; Advanced.** This is a self-contained offline exercise. A simulated rollout can illustrate why applying is not the final success criterion.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Copy the baseline before mutation. Restore on the known verification error and propagate the error. Follow the class-specific steps under Pattern 064 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
postcheck failure reported; local state restored
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This restores one dictionary only; it does not provide network-wide atomicity.

In [40]:
# Example 0640: Rollback a failed local transaction - backup
case = {'mtu': 8192}

class VerificationError(RuntimeError):
    pass

class LocalTransaction:
    def __init__(self):
        self.state = {"mtu": 1500}

    def apply(self, mtu, verify):
        before = self.state.copy()
        self.state["mtu"] = mtu
        try:
            if not verify():
                raise VerificationError("postcheck failed")
        except VerificationError:
            self.state = before
            raise

transaction = LocalTransaction()
try:
    transaction.apply(case["mtu"], lambda: False)
except VerificationError:
    assert transaction.state == {"mtu": 1500}
    print("postcheck failure reported; local state restored")
else:
    raise AssertionError("postcheck failure ignored")


postcheck failure reported; local state restored


## Pattern 065: Approval gate and explicit audit outcome

**Core concept:** Validation, approval, and audit are separate responsibilities.

**Network-engineering purpose:** A simulated change should record a denial instead of reporting that it applied successfully.

### Read the implementation in this order
1. Track the current value and audit events.
2. Require approval before mutation.
3. Record denied or applied explicitly.
4. Check the resulting state and outcome agree.

**Common mistake / boundary:** A boolean in a notebook is not a production identity, authorization, or approval system.

**Practice after the ten worked scenarios:** Bind approval to a digest of a specific reviewed plan and reject a changed plan.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0641 - Approval gate and explicit audit outcome: branch scenario

**Scenario 01/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `approved` | `True` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['applied'] 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [41]:
# Example 0641: Approval gate and explicit audit outcome - branch
case = {'mtu': 1500, 'approved': True}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['applied'] 1500


### Example 0642 - Approval gate and explicit audit outcome: campus scenario

**Scenario 02/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |
| `approved` | `False` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['denied'] 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [42]:
# Example 0642: Approval gate and explicit audit outcome - campus
case = {'mtu': 9000, 'approved': False}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['denied'] 1500


### Example 0643 - Approval gate and explicit audit outcome: dc scenario

**Scenario 03/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |
| `approved` | `True` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['applied'] 9216
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [43]:
# Example 0643: Approval gate and explicit audit outcome - dc
case = {'mtu': 9216, 'approved': True}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['applied'] 9216


### Example 0644 - Approval gate and explicit audit outcome: wan scenario

**Scenario 04/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |
| `approved` | `True` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['applied'] 576
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [44]:
# Example 0644: Approval gate and explicit audit outcome - wan
case = {'mtu': 576, 'approved': True}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['applied'] 576


### Example 0645 - Approval gate and explicit audit outcome: lab scenario

**Scenario 05/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |
| `approved` | `False` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['denied'] 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [45]:
# Example 0645: Approval gate and explicit audit outcome - lab
case = {'mtu': 1600, 'approved': False}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['denied'] 1500


### Example 0646 - Approval gate and explicit audit outcome: wireless scenario

**Scenario 06/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `approved` | `True` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['applied'] 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [46]:
# Example 0646: Approval gate and explicit audit outcome - wireless
case = {'mtu': 1500, 'approved': True}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['applied'] 1500


### Example 0647 - Approval gate and explicit audit outcome: remote scenario

**Scenario 07/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |
| `approved` | `False` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['denied'] 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [47]:
# Example 0647: Approval gate and explicit audit outcome - remote
case = {'mtu': 2000, 'approved': False}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['denied'] 1500


### Example 0648 - Approval gate and explicit audit outcome: core scenario

**Scenario 08/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |
| `approved` | `True` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['applied'] 4470
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [48]:
# Example 0648: Approval gate and explicit audit outcome - core
case = {'mtu': 4470, 'approved': True}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['applied'] 4470


### Example 0649 - Approval gate and explicit audit outcome: mgmt scenario

**Scenario 09/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |
| `approved` | `False` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['denied'] 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [49]:
# Example 0649: Approval gate and explicit audit outcome - mgmt
case = {'mtu': 9100, 'approved': False}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['denied'] 1500


### Example 0650 - Approval gate and explicit audit outcome: backup scenario

**Scenario 10/10; pattern 065; Advanced.** This is a self-contained offline exercise. A simulated change should record a denial instead of reporting that it applied successfully.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |
| `approved` | `True` |

**Logic to trace:** Track the current value and audit events. Check the resulting state and outcome agree. Follow the class-specific steps under Pattern 065 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['applied'] 8192
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A boolean in a notebook is not a production identity, authorization, or approval system.

In [50]:
# Example 0650: Approval gate and explicit audit outcome - backup
case = {'mtu': 8192, 'approved': True}

class ApprovedChange:
    def __init__(self):
        self.mtu, self.audit = 1500, []

    def apply(self, mtu, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        self.mtu = mtu
        self.audit.append("applied")

change = ApprovedChange()
try:
    change.apply(case["mtu"], approved=case["approved"])
except PermissionError:
    assert not case["approved"] and change.mtu == 1500
else:
    assert case["approved"] and change.mtu == case["mtu"]
assert change.audit == (["applied"] if case["approved"] else ["denied"])
print(change.audit, change.mtu)


['applied'] 8192


## Check your understanding: Desired state, idempotency, and rollback

**You should now be able to:** Separate intent, observation, plan, approval, application, verification, and recovery.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.